# NB03 — Data Analysis

**Question:** If I had invested in the S&P 500 ten years ago instead of leaving
the money in a savings account, what would the difference be today?



## Setup


In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# NOTE: not covered in course - plotly is used instead of matplotlib because the
# public page needs charts a reader can hover over to read exact monthly values.

df = pd.read_csv("../data/processed/monthly_returns.csv")

# One row per series per month -> pivot to one row per month, one column per series.
monthly = df.pivot(index="date", columns="series", values="return_pct")

print(monthly.shape)
print(monthly.index.min(), "to", monthly.index.max())
print(monthly.isna().sum())
monthly.head()

(120, 2)
2016-08 to 2026-07
series
BRMSA0104    0
SP500        0
dtype: int64


series,BRMSA0104,SP500
date,,
2016-08,0.006667,0.364226
2016-09,0.006667,-0.908904
2016-10,0.006667,-0.679893
2016-11,0.006667,1.024944
2016-12,0.006667,3.771080


## Chart style

Both charts below share one registered plotly template, so the house style is
defined once instead of being repeated per figure.

In [2]:
# One house style for every chart in this notebook, registered once as a plotly
# template, then two helpers for the furniture that repeats. The look follows the
# conventions of the financial press: warm off-white page, value axis on the
# right so the latest number sits where the eye stops, a rule above the headline,
# and the source named on the chart itself.
#
# Colours were checked for colour-vision deficiency before use. Deep blue and
# claret stay 16.2 apart under protanopia and 26.5 under tritanopia (OKLab
# distance), 25.5 for normal vision, and both clear 3:1 contrast against this
# page colour - the default plotly orange does not, and washed out on a light
# site background. Claret rather than a bright red on purpose: red would read as
# a warning, and the savings account never fell in any month of the window.

MARKET, SAVINGS = "#0f5499", "#990f3d"
PAPER = "#fffaf3"                       # warm off-white, as used in print
INK, MUTED, GRID = "#16232e", "#6b6459", "#e5ded4"
SERIF = "Georgia, Times New Roman, serif"
SANS = "Inter, -apple-system, Segoe UI, Helvetica, Arial, sans-serif"

pio.templates["me204"] = go.layout.Template(
    layout=dict(
        colorway=[MARKET, SAVINGS],
        font=dict(family=SANS, size=13, color=INK),
        # Centred on the whole image, not on the plotting area: the headline is
        # wider than the plot, so anchoring it to the paper pushed it off the
        # left edge and cut the first word off.
        title=dict(font=dict(family=SERIF, size=21, color=INK),
                   x=0.5, xanchor="center", xref="container", pad=dict(b=16)),
        paper_bgcolor=PAPER,
        plot_bgcolor=PAPER,
        # No axis lines: the gridlines and the baseline carry the structure.
        xaxis=dict(showgrid=False, showline=True, linecolor=INK, linewidth=1.2,
                   ticks="outside", tickcolor=GRID, ticklen=4,
                   tickfont=dict(size=12, color=MUTED),
                   title=dict(font=dict(size=11.5, color=MUTED))),
        yaxis=dict(side="right", showgrid=True, gridcolor=GRID, gridwidth=1,
                   zeroline=False, showline=False, ticks="",
                   tickfont=dict(size=12, color=MUTED),
                   title=dict(font=dict(size=11.5, color=MUTED))),
        legend=dict(orientation="h", y=1.0, x=0, yanchor="bottom",
                    font=dict(size=12, color=MUTED)),
        hoverlabel=dict(bgcolor="white", bordercolor=GRID,
                        font=dict(family=SANS, size=12, color=INK)),
        margin=dict(l=25, r=80, t=115, b=70),
    )
)
pio.templates.default = "plotly_white+me204"

SOURCE = ("Source: Federal Reserve Economic Data (FRED), series SP500 and "
          "BRMSA0104. Data retrieved 22 July 2026.")


def press_frame(fig, source=SOURCE):
    """Add the source line to the footer, centred under the plot.

    Paper coordinates run 0-1 over the plotting area, not the whole image, so the
    y position depends on the margins. Deriving it here means the line still lands
    correctly when a chart changes height - a hard-coded y did not.
    """
    plot_h = fig.layout.height - fig.layout.margin.t - fig.layout.margin.b

    fig.add_annotation(xref="paper", yref="paper",
                       x=0.5, y=-(fig.layout.margin.b - 30) / plot_h,
                       xanchor="center", yanchor="top", showarrow=False,
                       text=source, font=dict(size=10.5, color=MUTED))


def style_subplot_titles(fig, size=14):
    """make_subplots writes its titles as centred black sans annotations. Left-align
    them and give them the headline's serif in the subtitle's grey, so a panel label
    reads as part of the header hierarchy rather than as a stray line of text."""
    for note in fig.layout.annotations:
        note.update(x=0, xanchor="left",
                    font=dict(family=SERIF, size=size, color=MUTED))


def export(fig, name):
    """Write a figure to docs/ in both forms the public page can use: a
    self-contained HTML file that keeps the hover, and a PNG fallback.
    Regenerating the page assets is then just a matter of rerunning NB03."""
    path = Path(f"../docs/{name}.html")
    fig.write_html(path, include_plotlyjs="cdn", full_html=True,
                   config={"displayModeBar": False, "responsive": True})

    # plotly's page template keeps the browser's default 8px body margin. Inside
    # an iframe that pushes a 100%-wide chart past the edge, which raises a
    # horizontal scrollbar, which in turn raises a vertical one and hides part of
    # the figure. Zeroing it makes the frame height in the markdown exact.
    path.write_text(path.read_text().replace(
        "<head>",
        "<head><style>html,body{margin:0;padding:0;overflow:hidden}</style>", 1))

    fig.write_image(f"../docs/{name}.png", scale=2)

## Finding 1 - what $1,000 became



In [3]:
START = 1000    # invested once, in Aug 2016

growth = (1 + monthly / 100).cumprod() * START
growth.loc["2016-07"] = START     # both series start here, before the first month
growth = growth.sort_index()

x = [d + "-01" for d in growth.index]   # plotly parses these as dates
final = growth.iloc[-1]

fig = go.Figure()

# The two stretches worth naming, shaded before the lines are drawn.
for x0, x1, label in [("2020-02-01", "2020-08-01", "COVID crash<br>and recovery"),
                      ("2021-12-01", "2022-10-01", "2022<br>drawdown")]:
    fig.add_vrect(x0=x0, x1=x1, fillcolor="#f0e9dd", line_width=0, layer="below",
                  annotation_text=label, annotation_position="top left",
                  annotation_font=dict(size=10, color=MUTED))

for name, col, color, width in [("S&P 500", "SP500", MARKET, 2.4),
                                ("Savings account", "BRMSA0104", SAVINGS, 2)]:
    fig.add_trace(go.Scatter(
        x=x, y=growth[col], name=name, mode="lines",
        line=dict(color=color, width=width),
        hovertemplate="%{x|%b %Y}: $%{y:,.0f}<extra>" + name + "</extra>",
    ))
    # A tag past the right-hand axis, the way a price chart ends: the series is
    # named where its line stops, so no legend is needed at all.
    fig.add_annotation(
        xref="paper", x=1.075, y=final[col],
        text=f"<b>${final[col]:,.0f}</b><br>{name}",
        xanchor="left", align="left", showarrow=False,
        font=dict(color=color, size=12.5),
    )

fig.update_layout(
    title=dict(
        text=f"${START:,} invested in Aug 2016, and what it was worth by Jul 2026",
        # &#36; not $ - plotly treats a $...$ pair as LaTeX and would eat the text.
        subtitle=dict(
            text=f"The S&P 500 ended at &#36;{final['SP500']:,.0f}, the savings "
                 f"account at &#36;{final['BRMSA0104']:,.0f} \u2014 a gap of "
                 f"&#36;{final['SP500'] - final['BRMSA0104']:,.0f}.",
            font=dict(family=SANS, size=13, color=MUTED)),
    ),
    height=520, showlegend=False, hovermode="x unified",
    margin=dict(l=25, r=175, t=160, b=75),
)
fig.update_yaxes(title_text=None, tickprefix="$", tickformat=",.0f")
fig.update_xaxes(title_text=None, range=[x[0], x[-1]], dtick="M24", tickformat="%Y")

press_frame(fig)

export(fig, "finding1-growth")

fig.show()

## Finding 2 - year by year



In [4]:
annual = monthly.copy()
annual["year"] = [d[:4] for d in annual.index]
annual = annual.loc[~annual["year"].isin(["2016", "2026"])]

# Compound the months inside each year, then express the result in percent.
yearly = annual.groupby("year").apply(
    lambda g: (1 + g[["SP500", "BRMSA0104"]] / 100).prod() * 100 - 100,
    include_groups=False,
)

years = yearly.index.tolist()
won = (yearly["SP500"] > yearly["BRMSA0104"]).sum()
lost = [y for y in years if yearly.loc[y, "SP500"] < yearly.loc[y, "BRMSA0104"]]

fig = go.Figure()

# The connectors, drawn first so the dots sit on top of them. One trace with
# None between each pair keeps this to a single trace instead of nine shapes -
# and unlike shapes, a scatter trace places itself correctly on a category axis.
link_x, link_y = [], []
for yr in years:
    link_x += [yearly.loc[yr, "BRMSA0104"], yearly.loc[yr, "SP500"], None]
    link_y += [yr, yr, None]

fig.add_trace(go.Scatter(x=link_x, y=link_y, mode="lines", hoverinfo="skip",
                         line=dict(color="#ddd6cb", width=3), showlegend=False))

for name, col, color in [("Savings account", "BRMSA0104", SAVINGS),
                         ("S&P 500", "SP500", MARKET)]:
    fig.add_trace(go.Scatter(
        x=yearly[col], y=years, mode="markers", name=name,
        marker=dict(color=color, size=13, line=dict(color=PAPER, width=2)),
        hovertemplate="%{y}: %{x:.2f}%<extra>" + name + "</extra>",
    ))

# Name the two series once, above the top row, instead of carrying a legend.
# On a category axis an annotation is placed by position, not by the label, and
# the axis runs bottom-up, so the top row is the last position.
top = len(years) - 1
for name, col, color in [("S&P 500", "SP500", MARKET),
                         ("Savings account", "BRMSA0104", SAVINGS)]:
    fig.add_annotation(x=yearly.loc[years[-1], col], y=top, text=name,
                       showarrow=False, yshift=24,
                       font=dict(color=color, size=12))

# The two years the gap runs the other way.
for yr in lost:
    fig.add_annotation(x=yearly.loc[yr, "BRMSA0104"], y=years.index(yr),
                       text="savings ahead", showarrow=False,
                       xshift=14, xanchor="left",
                       font=dict(size=10, color=SAVINGS))

fig.update_layout(
    title=dict(
        text=f"Return by calendar year: the market was ahead in {won} of {len(years)} years",
        subtitle=dict(
            text="Each line is one year. Its length is what choosing the market "
                 "was worth that year - or, twice, what it cost.",
            font=dict(family=SANS, size=13, color=MUTED)),
    ),
    height=560, showlegend=False,
    margin=dict(l=25, r=45, t=160, b=105),
)
fig.update_yaxes(type="category", side="left",
                 showgrid=False, ticks="", title_text=None,
                 tickfont=dict(size=13, color=MUTED))
fig.update_xaxes(title_text="Return over the year (%)", ticksuffix="%",
                 showgrid=True, gridcolor=GRID, showline=False,
                 zeroline=True, zerolinecolor="#a8a095", zerolinewidth=1.2)

press_frame(fig)
export(fig, "finding2-by-year")

fig.show()

yearly.round(2)

series,SP500,BRMSA0104
year,,
2017,18.59,0.09
2018,-3.64,0.09
2019,23.74,0.10
2020,16.32,0.09
2021,26.51,0.06
2022,-16.31,0.11
2023,19.75,0.37
2024,28.30,0.52
2025,14.01,0.48
